# Kotoba Multilingual Speech-to-Text (STT) Streaming - Real-time Transcription in Eight Languages with Bidirectional Streaming

This notebook shows how to deploy and call the **Kotoba Multilingual Speech-to-Text (STT) Streaming** model package from AWS Marketplace, end-to-end.

**Product**: Kotoba Multilingual Speech-to-Text (STT) Streaming (SKU `kotoba-multilingual-stt-streaming`, version `v1.0`)
**Seller**: Kotoba AI

| | Kotoba Multilingual Speech-to-Text (STT) Streaming |
|---|---|
| Pricing | Per audio hour (one inference = 3600 s of decoded input audio) |
| Languages | `ja`, `en`, `zh`, `zh-tw` (Taiwan Traditional Chinese), `ko`, `yue`, `es`, `vi` |
| `turn_detection` when omitted | semantic VAD (`eagerness` `medium`) |
| `verbatim` when omitted | `true` (fillers kept); `false` is Japanese only |
| Server session limit | 24 hours |

## Key Features

- Real-time streaming transcription, text about 0.8 s behind the audio
- Eight languages; one language per session
- Multiple audio formats (`pcm16`, `float32`, `mulaw`, `opus`)
- Keyword biasing (up to 1000 terms); for Japanese, katakana person names and filler removal
- Turn detection on by default (semantic VAD), or server VAD, or off
- GPU-accelerated inference on NVIDIA L40S (`ml.g6e`, up to 72 sessions per instance), NVIDIA RTX PRO 6000 Blackwell (`ml.g7e`, up to 256) or NVIDIA H100 (`ml.p5.4xlarge`, up to 320)

Not available in this version: timestamps, speaker diarization.

See [README.md](./README.md) for the full list.

## Billing

**What is counted.** The length of the audio you send, after decoding. Connection time is not counted: an open stream that sends no audio adds no software charge.

| Case | Counted? |
| --- | --- |
| Audio you send in `input_audio_buffer.append`, silence included | Yes |
| Silence the server adds at `input_audio_buffer.commit` | No |
| An `append` that is rejected (for example, audio that cannot be decoded) | No |
| A stream that ends without `commit` (disconnect, expired credentials, the 30-minute limit) | Yes, for the audio already sent |
| An `append` that was decoded but that the server then failed to process (for example, an internal error) | Yes |

**How the listing shows the price.** The AWS Marketplace listing gives the price per "request" on the dimension `inference.count`. One request there is one inference: 3600 s (one hour) of audio. It is not one stream and not one event. Shorter audio is billed in proportion. For example, if the listing price is P per request, a stream that sends 6 minutes of audio is 0.1 inference and costs 0.1 × P. The listing gives the price.

**The other charge.** SageMaker bills the instance hours of the endpoint for as long as the endpoint runs, also when no audio is sent.

**Tip.** Silence you send is counted, so a client that stops sending during long pauses pays less. The model writes the text of speech only after it receives some audio that follows it (and, with turn detection, ends the turn on that audio). Before you stop sending, send silence until the text of the last speech has arrived, or send `input_audio_buffer.commit`.

---

## IMPORTANT: Bidirectional streaming only

Kotoba Multilingual Speech-to-Text (STT) Streaming uses **SageMaker bidirectional streaming exclusively**. It is **not** a standard REST endpoint:

- `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]==0.13.1` and `smithy-http[awscrt]` (this notebook uses the 0.13 client API)
- A bidirectional stream lasts at most **30 minutes**, and ends earlier when the AWS credentials that signed it expire. Open a new stream for longer audio.

If these constraints don't fit your workload, see the Troubleshooting section at the end.

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba Multilingual Speech-to-Text (STT) Streaming

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateModel",
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteModel",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:DescribeEndpointConfig",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Inference](#4.-Perform-Real-time-Inference)
5. [Troubleshooting](#5.-Troubleshooting)
6. [Clean-up](#6.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the model package:

1. Open the [listing of Kotoba Multilingual Speech-to-Text (STT) Streaming](https://aws.amazon.com/marketplace/pp/prodview-ven4djetklnes) on AWS Marketplace
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Install Dependencies

Run this cell first — the Python version check in 2.2 doesn't require these packages, but every cell after that does.

In [ ]:
# The crt extra is needed for credentials from `aws login`.
!pip install -q --upgrade "boto3[crt]" numpy soundfile
!pip install -q --upgrade "sagemaker>=2.0,<3"
# The HTTP/2 SDK is not yet 1.0 and changes its API between minor versions: this notebook
# uses 0.13. smithy-http[awscrt] adds the HTTP/2 transport that bidirectional streaming needs.
!pip install -q --upgrade "aws-sdk-python[sagemaker-runtime-http2]==0.13.1" "smithy-http[awscrt]"


### 2.2 Check Python Version

`aws-sdk-python` requires Python 3.12+. If this cell raises, switch your kernel (or create a new environment) before continuing.

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")


### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
from pathlib import Path

import boto3
import numpy as np
import sagemaker
import soundfile as sf

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.aio.client import AsyncSageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.aio.config import AsyncSageMakerRuntimeHTTP2Config
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from sagemaker import ModelPackage
from smithy_http.aio.crt import AWSCRTHTTPClient

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

SageMaker needs an **execution role** (an IAM *role*, not an IAM *user*) to assume when creating the endpoint.

- **On SageMaker Studio / Notebook Instance**: leave `SAGEMAKER_ROLE_ARN = None`. `sagemaker.get_execution_role()` will pick up the attached role automatically.
- **Running locally (IAM user / CLI profile)**: set `SAGEMAKER_ROLE_ARN` to the ARN of a role whose trust policy allows `sagemaker.amazonaws.com` to assume it and which has (at minimum) the `AmazonSageMakerFullAccess` managed policy.


In [ ]:
# --- Execution role -----------------------------------------------------
# If running on SageMaker Studio or a Notebook Instance, leave this as None.
# If running locally as an IAM user, set it to the ARN of a SageMaker-capable role, e.g.
#   SAGEMAKER_ROLE_ARN = "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
SAGEMAKER_ROLE_ARN: str | None = None
# ------------------------------------------------------------------------

sagemaker_session = sagemaker.Session()
region = sagemaker_session.boto_region_name

if SAGEMAKER_ROLE_ARN is not None:
    role = SAGEMAKER_ROLE_ARN
else:
    try:
        role = sagemaker.get_execution_role()
    except ValueError as exc:
        raise ValueError(
            "get_execution_role() only works in SageMaker-managed environments. "
            "Set SAGEMAKER_ROLE_ARN above to a SageMaker execution role ARN "
            "(not your IAM user ARN). See the markdown cell above for role requirements."
        ) from exc

print(f"Region: {region}")
print(f"Role:   {role}")


### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# Replace the placeholder with the Model Package ARN of your AWS Marketplace subscription (section 1, step 5).
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )


## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

Supported instance types:

| Instance | GPU | Concurrent sessions per instance | Notes |
|----------|-----|----------------------------------|-------|
| `ml.g6e.4xlarge`, `ml.g6e.8xlarge`, `ml.g6e.16xlarge` | 1x NVIDIA L40S | up to 72 (measured on `ml.g6e.4xlarge`) | `ml.g6e.4xlarge` is recommended |
| `ml.g7e.4xlarge`, `ml.g7e.8xlarge` | 1x NVIDIA RTX PRO 6000 Blackwell | up to 256 | |
| `ml.p5.4xlarge` | 1x NVIDIA H100 | up to 320 | |

The larger sizes of a family add CPU and memory, not GPUs. Size for your peak number of concurrent sessions. Smaller `ml.g6e` sizes are not supported: their CPU cannot keep up at this load.

The first start takes about 5 minutes (the model is loaded and compiled).

In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-multilingual-stt-{int(time.time())}"
INSTANCE_TYPE = "ml.g6e.4xlarge"  # or ml.g6e.8xlarge / ml.g6e.16xlarge (L40S), ml.g7e.4xlarge / ml.g7e.8xlarge (RTX PRO 6000), ml.p5.4xlarge (H100)

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")


In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint (this takes 5-10 minutes)
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is InService.")


## 4. Perform Real-time Inference

### 4.1 Connection Protocol

Kotoba Multilingual Speech-to-Text (STT) Streaming is reached through SageMaker's bidirectional-streaming transport:

```
Client (HTTP/2 SDK) -> SageMaker Runtime (HTTPS:8443) -> Container
```

Standard WebSocket libraries **cannot** connect to SageMaker endpoints directly — you must use `aws-sdk-python[sagemaker-runtime-http2]`.

### 4.2 Audio Format

| Format | Sample rates (Hz) | Channels | Encoding |
|--------|-------------------|----------|----------|
| `pcm16`   | any (24000 recommended) | 1 or more (downmixed to mono) | Little-endian int16 |
| `float32` | any (24000 recommended) | 1 or more (downmixed to mono) | Little-endian float32, range `[-1.0, 1.0]` |
| `mulaw`   | 8000 | 1 (mono) | G.711 μ-law (`twilio` is an alias) |
| `opus`    | decoded at 48000 | 1 (mono) | Self-contained Ogg/Opus per `append` |

- Default sample rate: **24000 Hz**. Other rates are resampled server-side.
- Recommended chunk size: **80 ms** per `input_audio_buffer.append` event.
- Maximum audio per `input_audio_buffer.append`: **8 seconds**.

### 4.3 Protocol Flow

```
Client                                                              Server
  |<----- transcription_session.created ----------------------------|
  |------ transcription_session.update ---------------------------->|
  |<----- transcription_session.updated ----------------------------|
  |<----- conversation.item.created --------------------------------|   (emitted once, before first delta)
  |------ input_audio_buffer.append ------------------------------->|   (repeated, streaming)
  |<----- conversation.item.input_audio_transcription.delta --------|   (repeated)
  |<----- conversation.item.input_audio_transcription.completed ----|   (per turn, if turn_detection on)
  |------ input_audio_buffer.commit ------------------------------->|
  |<----- input_audio_buffer.committed -----------------------------|   (then the server closes the stream)
```

### 4.4 Event Reference

#### Client Events (you send)

| Event | Purpose |
|-------|---------|
| `transcription_session.update` | Configure the session (audio format, language, optional keywords). Send **once** after `transcription_session.created`. |
| `input_audio_buffer.append`    | Send one audio chunk (base64-encoded bytes). Repeat as audio arrives. |
| `input_audio_buffer.commit`    | Signal "no more audio." Triggers final flush on the server. Optional `event_id`, returned as `item_id` on `committed`. |

#### Server Events (you receive)

| Event | Purpose |
|-------|---------|
| `transcription_session.created` | Session is ready — send your `transcription_session.update` next. |
| `transcription_session.updated` | Server accepted your config. Safe to start streaming audio. |
| `conversation.item.created`     | A conversation item was opened to hold the incoming transcript. Informational — no action needed. |
| `conversation.item.input_audio_transcription.delta` | Transcription fragment. Concatenate `delta` fields in order. Carries `audio_processed_ms`, which stays `0` on this product. |
| `conversation.item.input_audio_transcription.completed` | Only when turn detection is enabled (see the note below). One per detected turn, carrying the turn's `item_id` and full `transcript`. |
| `input_audio_buffer.committed`  | All audio processed (`audio_processed_ms` = length of the processed audio). The server then closes the stream. |
| `error`                          | Something went wrong. Payload: `{"type": "error", "event_id": ..., "error": {...}, "item_id"?: ...}`. |

> **Note**: on this product `turn_detection` defaults to semantic VAD (`eagerness` `medium`); send `false` to turn it off. Set it to a semantic VAD object (`{ "type": "semantic_vad", "eagerness": "auto" }`, recommended) or a server VAD object (`{ "type": "server_vad", "silence_duration_ms": 800 }`) to enable automatic turn detection. The server then emits a `conversation.item.input_audio_transcription.completed` event (with `item_id` and the turn's full `transcript`) at each turn boundary, and the turn's deltas carry the same server-generated `item_id`. `eagerness` (`low`, `medium`, `high`, `auto` = `medium`) sets how soon a semantic VAD turn ends. `silence_duration_ms` is the time without text output that ends a server VAD turn; it **defaults to 800 ms but should be tuned for your application and the pipeline it is embedded in**, not left at the default. With `false`, rely on the concatenation of `delta` events plus `input_audio_buffer.committed` as the "done" signal.

See [data/sample_input.json](./data/sample_input.json) and [data/sample_output.json](./data/sample_output.json) for canonical event payloads.

### 4.4b Server-side Timeouts and Stream Limits

| Phase | Server behavior |
|-------|-----------------|
| Session init | A valid `transcription_session.update` must arrive within **60 s** of connecting. An invalid update is answered with an `error` event (`invalid_parameters`) and the server keeps waiting. After 60 s the server sends an `error` event (`timeout_error`) and closes the stream. |
| After commit | The server processes the remaining audio (at most **30 s**), sends `input_audio_buffer.committed` and closes the stream. |
| Whole stream | At most **30 minutes** ([AWS documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)), and the stream ends when the AWS credentials that signed it expire. No `error` event is sent: the output stream just ends. |

In between — once `transcription_session.updated` has been received — there is
**no server-side inactivity or result timeout**, and no `timeout` event in the STT
protocol. A session that completed init but never receives
`input_audio_buffer.append` stays open until the client or the stream limit closes
it, so always enforce your own client-side read timeout.

For audio longer than one stream, or after a disconnect, open a new stream and resend the audio whose text you have not received. `input_audio_buffer.committed` carries `audio_processed_ms`, the length of the processed audio; on this product the deltas carry `0`.

### 4.5 Prepare Input Audio

Two paths:

- **Smoke test** (next cell): sends 3 seconds of silence. Useful purely to confirm the endpoint is reachable and the protocol round-trips. The transcript will be empty.
- **Real audio** (cell after): loads a WAV file. Use this to see an actual transcription.


In [ ]:
# --- Connectivity smoke test (silent audio) ---------------------------------
# Emits 3 s of silence, then runs the full transcription_session ->
# input_audio_buffer.commit flow. Expect an EMPTY transcript; the goal is to
# confirm the endpoint routes events correctly end-to-end.

SAMPLE_RATE = 24000
CHUNK_DURATION_MS = 80  # recommended chunk size

smoke_test_samples = np.zeros(SAMPLE_RATE * 3, dtype=np.int16)
smoke_test_bytes = smoke_test_samples.tobytes()
SMOKE_TEST_LANGUAGE = "ja"

print(f"Smoke-test audio ready: {len(smoke_test_bytes)} bytes "
      f"({len(smoke_test_samples) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz, pcm16)")


### 4.5b Load Your Own Audio (recommended)

Replace `WAV_PATH` with any audio file readable by [`soundfile`](https://pysoundfile.readthedocs.io/) (WAV / FLAC / OGG / ...). The cell below checks that the file is mono, decodes it to int16 and prepares it for streaming. The server resamples any rate; 24 kHz is recommended.

```python
WAV_PATH = "path/to/your_audio.wav"
AUDIO_LANGUAGE = "ja"
```

In [ ]:
# ----- Replace THIS with your own audio file -------------------------------
WAV_PATH = "path/to/audio"   # any format supported by libsndfile (WAV, FLAC, OGG, ...)
AUDIO_LANGUAGE = "ja"                   # "ja", "en", "zh", "zh-tw", "ko", "yue", "es" or "vi"
# ---------------------------------------------------------------------------


def load_pcm16(path: str) -> tuple[bytes, int]:
    """Load an audio file and return (int16 little-endian PCM bytes, sample_rate).

    Accepts any format `soundfile` can decode (WAV, FLAC, OGG, etc.). Requires mono;
    the server resamples any sample rate.
    """
    data, samplerate = sf.read(path, dtype="int16")
    if data.ndim != 1:
        raise ValueError(
            f"{path}: expected mono, got {data.shape[1]} channels. "
            "Mix down to mono before passing to Kotoba Multilingual Speech-to-Text (STT) Streaming."
        )
    return data.tobytes(), samplerate


if Path(WAV_PATH).exists():
    real_audio_bytes, real_audio_rate = load_pcm16(WAV_PATH)
    print(
        f"Loaded {WAV_PATH}: {len(real_audio_bytes)} bytes "
        f"({len(real_audio_bytes) / 2 / real_audio_rate:.2f} s @ {real_audio_rate} Hz)"
    )
else:
    real_audio_bytes = None
    real_audio_rate = None
    print(
        f"WAV_PATH={WAV_PATH!r} not found. Update the path above or skip this cell "
        "and run the smoke test only."
    )


### 4.6 Streaming Inference

It establishes the HTTP/2 stream, waits for `transcription_session.created` (with a timeout),
sends configuration, streams audio chunks with real-time pacing, and concatenates
`delta` events into the full transcript.

### Session configuration options

The `session` object inside `transcription_session.update` supports:

| Field | Type | Default | Notes |
|-------|------|---------|-------|
| `input_audio_format` | `"pcm16"` \| `"float32"` \| `"mulaw"` \| `"opus"` | `"pcm16"` | Must match the bytes you send. `"twilio"` is an alias of `"mulaw"`. |
| `input_audio_sample_rate` | int | `24000` | Any positive rate; the server resamples. |
| `input_audio_number_of_channels` | int | `1` | PCM input with more channels is downmixed to mono. |
| `input_audio_transcription.target_language` | str | *(required)* | `"ja"`, `"en"`, `"zh"`, `"zh-tw"`, `"ko"`, `"yue"`, `"es"` or `"vi"` |
| `input_audio_transcription.language` | str | `None` | If set, must equal `target_language`. |
| `input_audio_transcription.keywords` | list[str] | `None` | Bias recognition toward the listed terms. Up to 1000 terms, each up to 64 characters; over either limit returns an `invalid_parameters` error. |
| `input_audio_transcription.style_preference.human_name` | `"kana"` \| `"default"` | `"default"` | `"kana"` transcribes Japanese person names in katakana. |
| `input_audio_transcription.style_preference.verbatim` | bool | `True` | `False` omits fillers and restarts (e.g. えーと, あの) from **Japanese** transcripts; with any other language it is rejected. |
| `input_audio_transcription.timestamps` | bool | `False` | Not available; `True` is rejected with `invalid_parameters`. |
| `turn_detection` | `false` or object | semantic VAD | Omitted = `{ "type": "semantic_vad", "eagerness": "medium" }`; semantic VAD is the recommended type. `false` disables turn detection. A server VAD object (`{ "type": "server_vad", "silence_duration_ms": 800 }`) is also accepted. With turn detection on, the server emits `conversation.item.input_audio_transcription.completed` per turn. |
| `turn_detection.eagerness` | str | `"medium"` | Semantic VAD: `low`, `medium`, `high` or `auto` (= `medium`). Higher ends turns sooner. |
| `turn_detection.silence_duration_ms` | int | `800` | Server VAD: time without text output (ms) that ends a turn. **Tune for your application and the pipeline it is embedded in.** |
| `diarization` | bool | `False` | Not available; `True` is rejected with `invalid_parameters`. |

Semantic VAD ends a turn soon after a sentence is complete and waits longer while it is not; server VAD ends a turn after every pause of `silence_duration_ms`, which can split a phone number or an address that is read with pauses. For dictated numbers and addresses, use semantic VAD or a longer `silence_duration_ms`.

In [ ]:
SESSION_INIT_TIMEOUT = 10.0   # seconds to wait for transcription_session.created/updated (the server allows 60 s)
SESSION_COMMIT_TIMEOUT = 30.0  # seconds to wait for input_audio_buffer.committed after the commit
CHUNK_SAMPLES_PCM16 = int(SAMPLE_RATE * CHUNK_DURATION_MS / 1000)
CHUNK_BYTES_PCM16 = CHUNK_SAMPLES_PCM16 * 2  # 2 bytes per int16 sample


async def transcribe(
    endpoint_name: str,
    audio_bytes: bytes,
    *,
    sample_rate: int = SAMPLE_RATE,
    language: str = "ja",
    client_id: str = "nb",
    keywords: list[str] | None = None,
    verbatim: bool | None = None,
    turn_detection: dict | bool | None = None,
) -> str:
    """Transcribe pcm16 mono audio via Kotoba Multilingual Speech-to-Text (STT) Streaming.

    `verbatim=False` (Japanese only) removes fillers. `turn_detection=None` keeps the
    product default (semantic VAD); pass False to turn it off.
    """

    credentials = boto3.Session().get_credentials().get_frozen_credentials()
    config = await AsyncSageMakerRuntimeHTTP2Config.resolve(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        transport=AWSCRTHTTPClient(),  # HTTP/2, needed for bidirectional streaming
    )
    client = AsyncSageMakerRuntimeHTTP2Client(config=config)

    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )

    # NOTE: data_type="UTF8" is required. Without it the SageMaker runtime
    # relays our payloads as WebSocket *binary* frames, but the server calls
    # starlette's websocket.receive_json() which only accepts text frames
    # (it internally reads message["text"]). The flag tells SageMaker the
    # bytes are UTF-8 JSON so the frame is delivered as text.
    async def send_event(payload: dict) -> None:
        body = json.dumps(payload).encode("utf-8")
        await stream.input_stream.send(
            RequestStreamEventPayloadPart(
                value=RequestPayloadPart(bytes_=body, data_type="UTF8")
            )
        )

    _, output_stream = await stream.await_output()

    async def next_event() -> dict:
        message = await output_stream.receive()
        if message is None:
            raise RuntimeError("Stream closed unexpectedly")
        return json.loads(message.value.bytes_.decode("utf-8"))

    # 1) Wait for transcription_session.created
    async with asyncio.timeout(SESSION_INIT_TIMEOUT):
        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during init: {event}")
        assert event["type"] == "transcription_session.created", event
        print("[session] created")

        # 2) Send session configuration
        await send_event(
            {
                "type": "transcription_session.update",
                "session": {
                    "input_audio_format": "pcm16",
                    "input_audio_sample_rate": sample_rate,
                    "input_audio_number_of_channels": 1,
                    "input_audio_transcription": {
                        "language": language,
                        "target_language": language,
                        "keywords": keywords or [],
                        **({"style_preference": {"verbatim": verbatim}} if verbatim is not None else {}),
                    },
                    **({"turn_detection": turn_detection} if turn_detection is not None else {}),
                },
            }
        )

        # 3) Wait for transcription_session.updated
        event = await next_event()
        if event["type"] == "error":
            raise RuntimeError(f"Server error during update: {event}")
        assert event["type"] == "transcription_session.updated", event
        print("[session] updated")

    # 4) Stream audio chunks (sender) while collecting deltas (receiver)
    transcript_parts: list[str] = []
    chunk_bytes = int(sample_rate * CHUNK_DURATION_MS / 1000) * 2  # pcm16 = 2 bytes/sample

    async def send_audio() -> None:
        for i in range(0, len(audio_bytes), chunk_bytes):
            chunk = audio_bytes[i : i + chunk_bytes]
            await send_event(
                {
                    "event_id": f"{client_id}_{i // chunk_bytes:07d}",
                    "type": "input_audio_buffer.append",
                    "audio": base64.b64encode(chunk).decode(),
                }
            )
            await asyncio.sleep(CHUNK_DURATION_MS / 1000)  # simulate real-time pacing
        await send_event({"type": "input_audio_buffer.commit"})

    async def receive_deltas() -> None:
        audio_seconds = len(audio_bytes) / 2 / sample_rate
        async with asyncio.timeout(audio_seconds + SESSION_COMMIT_TIMEOUT):
            while True:
                event = await next_event()
                etype = event["type"]
                if etype == "conversation.item.input_audio_transcription.delta":
                    delta = event.get("delta", "")
                    transcript_parts.append(delta)
                    print(delta, end="", flush=True)
                elif etype == "conversation.item.created":
                    continue  # informational
                elif etype == "input_audio_buffer.committed":
                    print()  # newline after streaming deltas
                    return
                elif etype == "error":
                    raise RuntimeError(f"Server error: {event}")

    await asyncio.gather(send_audio(), receive_deltas())
    await stream.input_stream.close()

    return "".join(transcript_parts)


### 4.7 Run Transcription

**Smoke test first** — confirms the endpoint is healthy. Expect an empty transcript.

In [ ]:
smoke_transcript = await transcribe(
    endpoint_name=ENDPOINT_NAME,
    audio_bytes=smoke_test_bytes,
    sample_rate=SAMPLE_RATE,
    language=SMOKE_TEST_LANGUAGE,
    client_id="smoke",
)
print(f"\nSmoke-test transcript (expected empty): {smoke_transcript!r}")


**Now the real audio** — run this only after the smoke test above completes successfully.

In [ ]:
if real_audio_bytes is None:
    print("Skipping: real audio not loaded. Set WAV_PATH in the loader cell and re-run.")
else:
    transcript = await transcribe(
        endpoint_name=ENDPOINT_NAME,
        audio_bytes=real_audio_bytes,
        sample_rate=real_audio_rate,
        language=AUDIO_LANGUAGE,
        client_id="nb",
    )
    print(f"\n--- Full transcript ---\n{transcript}")


## 5. Troubleshooting

| Symptom | Likely cause & fix |
|---------|--------------------|
| `RuntimeError: Python 3.12+ is required` | Your kernel is < 3.12. Create a new environment with 3.12+ and re-attach the kernel. |
| `ModuleNotFoundError: aws_sdk_sagemaker_runtime_http2` | The HTTP/2 SDK install failed. Re-run cell 2.1. On SageMaker Studio, make sure the kernel is restarted after install. |
| `ImportError` from `aws_sdk_sagemaker_runtime_http2` (for example `cannot import name 'AsyncSageMakerRuntimeHTTP2Config'`) | Another version of the HTTP/2 SDK is installed; this notebook needs 0.13. Re-run cell 2.1, which pins it, and restart the kernel. |
| `UnsupportedTransportError`: the transport `AIOHTTPClient` does not support duplex event streaming | The client was built without the HTTP/2 transport. Keep `transport=AWSCRTHTTPClient()` in the config, and re-run cell 2.1 so that `smithy-http[awscrt]` is installed. |
| `TimeoutError` waiting for `transcription_session.created` | Endpoint is not yet `InService`, the endpoint name is wrong, or the region in `boto3.Session()` doesn't match the endpoint's region. Verify with `sagemaker_session.describe_endpoint(EndpointName=ENDPOINT_NAME)`. |
| `TimeoutError` waiting for `input_audio_buffer.committed` | Model is overloaded, or the posted audio is too long to process within `SESSION_COMMIT_TIMEOUT`. Raise the timeout or split long audio into shorter sessions. |
| `Server error: {'type': 'error', 'error': {...}}` | Server-side validation failure; `error.param` names the field. Common causes: `target_language` not "ja", "en", "zh", "zh-tw", "ko", "yue", "es", "vi", `timestamps: true`, more than 1000 keywords or a keyword over 64 characters, `diarization: true`, an append over 8 seconds of audio. |
| `invalid_parameters` with `error.param` `session.diarization` | This model does not separate speakers. Remove `diarization` or send `false`. |
| Transcript is empty on real audio | Double-check `AUDIO_LANGUAGE` matches the spoken language, confirm the WAV is mono 16-bit PCM at a supported rate, and verify the audio is actually audible (not silence). |
| `AccessDeniedException` at deploy time | Your IAM role lacks the permissions listed in section "Required IAM Permissions" above, or you haven't completed the AWS Marketplace subscription. |
| Session opens but sits idle until your client read-timeout fires (e.g. AWS `ReadTimeoutException`) | Expected when no audio is sent: after `transcription_session.updated` the STT server has **no inactivity timeout** and never closes an established session on its own (see 4.4b). Enforce your own client-side timeout and close idle sessions. |
| Connection closes ~60 s after connect, before `transcription_session.updated` | No valid `transcription_session.update` arrived within the server's **60 s** init window — the server sends an `error` (`timeout_error`) and closes. Send it immediately after `transcription_session.created`, and fix any field an earlier `invalid_parameters` error named. |
| Connection closes right after `input_audio_buffer.committed` | Expected — the session ends at `committed` and the server closes the stream. Open a new stream for the next audio. |
| `error` with `error.code` `system_busy`, or the stream is refused | The instance is at its session capacity. Retry with backoff, or add instances. |
| Stream ends after a few minutes with no `error` event | The AWS credentials that signed the stream expired (for example a 15-minute `aws login` or SSO token). Open each stream with credentials that outlive the session, or open a new stream before they expire. SageMaker's own limit is 30 minutes. |
| Transcript keeps fillers (えーと, あの) | Send `style_preference.verbatim: false` (Japanese only). |
| `invalid_parameters` with `error.param` `session.input_audio_transcription` and `verbatim: false` | `verbatim: false` works for `target_language` `"ja"` only. |
| `server_error` "Internal server error" a few seconds into a session, after which the stream closes | Audio was sent faster than real time: more than about 8 s ahead fills the server's buffer and ends the session. Pace appends in real time (the `transcribe()` function above sleeps 80 ms per chunk), and do not send a backlog all at once after a client-side pause. |


## 6. Clean-up

**Important**: Delete the endpoint when done. GPU instances are billed **per hour while the endpoint is active**, even when idle. The audio you stream is also billed per audio hour.

In [ ]:
sagemaker_client = sagemaker_session.sagemaker_client

# Look up the endpoint configuration and the model, so this cell works after a kernel restart too.
config_name = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
model_names = [
    variant["ModelName"]
    for variant in sagemaker_client.describe_endpoint_config(EndpointConfigName=config_name)[
        "ProductionVariants"
    ]
]

print(f"Deleting endpoint '{ENDPOINT_NAME}', endpoint config '{config_name}' and model(s) {model_names}...")
sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(config_name)
for model_name in model_names:
    sagemaker_session.delete_model(model_name)

print("Endpoint, endpoint config and model deleted.")


### Unsubscribe from Model Package (Optional)

To unsubscribe from the model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba Multilingual Speech-to-Text (STT) Streaming" in your subscriptions
3. Click **Manage** -> **Actions** -> **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- Canonical protocol samples: [data/sample_input.json](./data/sample_input.json), [data/sample_output.json](./data/sample_output.json), [README.md](./README.md)
- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)